# Data Prep

This notebook builds the launch-level base table that will be used for future feature engineering and modeling work.

Scope:

- load the raw launch-side tables from `data_final`
- clean key numeric and timestamp fields
- join launch records to location, mission, configuration, family, and company metadata
- standardize facility naming and facility coordinates
- join in whatever NOAA weather is currently available
- produce one cleaned dataframe that is ready for the next stage of feature engineering

The NOAA weather section uses the local LCDv2 documentation file
`lcdv2_DOCUMENTATION.pdf` as the reference for how the LCD weather tables are structured.
In practical terms, the join logic here assumes the downloaded NOAA files mix hourly,
daily, and other report types, so the notebook keeps the strongest hourly observation at
each timestamp and then matches launches to the nearest hourly weather record within a
2-hour tolerance.

In [22]:
from pathlib import Path

import numpy as np
import pandas as pd

# All raw and derived project data now lives under data_final.
DATA_DIR = Path("data_final")
OUTPUT_DIR = DATA_DIR / "derived"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

## Load Raw Tables

This first pass is only about getting the core source tables into memory and checking that
the current project inputs are the ones we expect. Keeping this summary visible makes it
easier to notice if a source file changes shape before we start joining everything
together.

In [23]:
# Launches is the row-level backbone. The remaining tables add metadata at different
# levels of detail: location/site, mission manifest, vehicle configuration, family, and
# organization.
launches = pd.read_csv(DATA_DIR / "Launches.csv")
locations = pd.read_csv(DATA_DIR / "Locations.csv")
missions = pd.read_csv(DATA_DIR / "Missions.csv")
configs = pd.read_csv(DATA_DIR / "Configs.csv")
families = pd.read_csv(DATA_DIR / "Families.csv")
companies = pd.read_csv(DATA_DIR / "Companies.csv")

raw_table_summary = pd.DataFrame(
    [
        {"table": "Launches", "rows": len(launches), "columns": launches.shape[1]},
        {"table": "Locations", "rows": len(locations), "columns": locations.shape[1]},
        {"table": "Missions", "rows": len(missions), "columns": missions.shape[1]},
        {"table": "Configs", "rows": len(configs), "columns": configs.shape[1]},
        {"table": "Families", "rows": len(families), "columns": families.shape[1]},
        {"table": "Companies", "rows": len(companies), "columns": companies.shape[1]},
    ]
)

raw_table_summary

,table,rows,columns
0,Launches,6168,16
1,Locations,145,14
2,Missions,7450,4
3,Configs,480,13
4,Families,205,8
5,Companies,59,3


## Cleaning Helpers

These helper functions centralize the project-specific cleanup rules that get reused across
multiple joins:

- `parse_numeric_text` converts text fields such as `"4,940 kN"` or `"13.0 m"` into numeric
  values that can be modeled later.
- `infer_facility_group` standardizes location naming so launches from the same operational
  facility land in one group even when the raw launch strings differ pad by pad.

In [24]:
def parse_numeric_text(series: pd.Series) -> pd.Series:
    # Many config fields are stored as strings with units, commas, or symbols.
    # We extract the numeric component so later feature engineering can work with real
    # numeric values instead of ad hoc text parsing in downstream notebooks.
    return pd.to_numeric(
        series.astype(str).str.extract(r"([-+]?[0-9]*\.?[0-9]+)")[0],
        errors="coerce",
    )


def infer_facility_group(row) -> str:
    # For non-U.S. launches, the combined launch site from Locations.csv is already the
    # right grouping level. For U.S. sites, we intentionally normalize to the facility
    # names used elsewhere in the NOAA workflow so downstream joins stay consistent.
    location = str(row.get("Location", ""))
    country_code = str(row.get("Country_Code", "")).upper()
    combined = row.get("Comb Launch Site")

    if country_code != "US":
        return combined

    text = location.lower()
    if "kennedy space center" in text:
        return "Kennedy Space Center"
    if "cape canaveral" in text:
        return "Cape Canaveral Space Force Station"
    if "vandenberg" in text:
        return "Vandenberg Space Force Base"
    if "wallops" in text:
        return "Wallops Flight Facility"
    if "pacific spaceport" in text or "kodiak" in text:
        return "Pacific Spaceport Complex Alaska"
    if "china lake" in text:
        return "China Lake"
    if "edwards" in text:
        return "Edwards Air Force Base"
    if "mojave" in text:
        return "Mojave Air and Space Port"
    if "kauai" in text or "pacific missile range" in text:
        return "Pacific Missile Range Facility"
    return combined

## Prepare Join Tables

Before building the final launch-level dataframe, we clean each auxiliary table into a
join-ready form:

- `mission_agg` collapses `Missions.csv` to one row per launch
- `config_features` combines vehicle configuration attributes with family-level metadata
- `company_features` standardizes organization metadata for a clean join
- `location_features` narrows `Locations.csv` to the columns needed in the base table

Doing this up front keeps the final launch-table construction easier to read and easier to
debug.

In [25]:
# Missions can have multiple rows per launch, so we aggregate them first to avoid
# duplicating launch rows later.
mission_agg = (
    missions.copy()
    .assign(
        Launch_Id=lambda df: pd.to_numeric(df["Launch Id"], errors="coerce"),
        mission_row_no=lambda df: pd.to_numeric(df["No"], errors="coerce"),
        payload_count_component=lambda df: pd.to_numeric(df["Payloads"], errors="coerce"),
        mission_mass_component=lambda df: pd.to_numeric(df["Mass"], errors="coerce"),
    )
    .groupby("Launch_Id", dropna=False)
    .agg(
        payload_count=("payload_count_component", "sum"),
        mission_mass=("mission_mass_component", "sum"),
        mission_rows=("mission_row_no", "count"),
    )
    .reset_index()
    .rename(columns={"Launch_Id": "Launch Id"})
)

# Configs carries vehicle-level engineering attributes, while Families adds the broader
# rocket family label and historical family success rate.
config_features = configs.merge(
    families[["Family Id", "Family", "Success Rate"]],
    on="Family Id",
    how="left",
).copy()

for col in [
    "Price",
    "Liftoff Thrust",
    "Payload to LEO",
    "Payload to GTO",
    "Stages",
    "Strap-ons",
    "Rocket Height",
    "Fairing Diameter",
    "Fairing Height",
]:
    if col in config_features.columns:
        config_features[col] = parse_numeric_text(config_features[col])

# Family success rate is stored as a percentage string such as "98.2%".
config_features["family_success_rate_pct"] = pd.to_numeric(
    config_features["Success Rate"].astype(str).str.rstrip("%"),
    errors="coerce",
)

config_features = config_features.rename(
    columns={
        "Config": "config_name",
        "Status": "config_status",
        "Price": "config_price_musd_text_parsed",
        "Liftoff Thrust": "config_liftoff_thrust",
        "Payload to LEO": "config_payload_leo",
        "Payload to GTO": "config_payload_gto",
        "Stages": "config_stages",
        "Strap-ons": "config_strap_ons",
        "Rocket Height": "config_rocket_height",
        "Fairing Diameter": "config_fairing_diameter",
        "Fairing Height": "config_fairing_height",
        "Family": "rocket_family",
    }
)

# Companies is already close to join-ready; we only rename columns into the launch-table
# naming style we want to keep downstream.
company_features = companies.rename(
    columns={
        "Company Name": "Rocket Organisation",
        "Company Country": "company_country",
        "Ownership": "company_ownership",
    }
).copy()

# Locations.csv contains both raw pad coordinates and combined-site coordinates. We keep
# both because they are useful for later feature engineering and NOAA join diagnostics.
location_features = locations[
    [
        "Orig_Addr",
        "Country",
        "Country_Code",
        "Lat",
        "Lon",
        "Operator",
        "Launch Site",
        "Launch Site Lat",
        "Launch Site Lon",
        "Comb Launch Site",
        "Comb Launch Site Lat",
        "Comb Launch Site Lon",
        "Operator Lat",
        "Operator Lon",
    ]
].copy()

## Build Clean Launch-Level Base Table

This is the main launch-data assembly step.

Why this step matters:

- it converts time fields into a consistent UTC representation
- it joins in site, mission, rocket, and company metadata
- it creates a standardized `facility_group`
- it preserves both raw and grouped spatial information for later modeling and NOAA joins

The result is the clean launch-side table that everything else in the notebook builds on.

In [26]:
# Start from the raw launch table and progressively add cleaned time fields plus joined
# metadata from the supporting tables.
base_modeling_df = launches.copy()

base_modeling_df["Launch Id"] = pd.to_numeric(base_modeling_df["Launch Id"], errors="coerce")
base_modeling_df["launch_time_utc"] = pd.to_datetime(base_modeling_df["Launch Time"], utc=True, errors="coerce")
base_modeling_df["launch_date"] = base_modeling_df["launch_time_utc"].dt.date
base_modeling_df["launch_year"] = base_modeling_df["launch_time_utc"].dt.year
base_modeling_df["launch_month"] = base_modeling_df["launch_time_utc"].dt.month
base_modeling_df["launch_month_name"] = base_modeling_df["launch_time_utc"].dt.month_name()
base_modeling_df["launch_quarter"] = base_modeling_df["launch_time_utc"].dt.quarter
base_modeling_df["launch_dayofweek"] = base_modeling_df["launch_time_utc"].dt.dayofweek
base_modeling_df["launch_hour_utc"] = base_modeling_df["launch_time_utc"].dt.hour
base_modeling_df["launch_decade"] = (base_modeling_df["launch_year"] // 10) * 10

for col in [
    "Rocket Price",
    "Rocket Payload to LEO",
    "USD/kg to LEO",
    "2021 Mult",
    "USD/kg to LEO CPI Adjusted",
    "Rocket Price CPI Adjusted",
]:
    if col in base_modeling_df.columns:
        base_modeling_df[col] = pd.to_numeric(base_modeling_df[col], errors="coerce")

# Join site/location metadata first because facility grouping depends on these fields.
base_modeling_df = base_modeling_df.merge(
    location_features,
    left_on="Location",
    right_on="Orig_Addr",
    how="left",
)
# Join mission aggregates so each launch row gets payload count and mission mass summaries.
base_modeling_df = base_modeling_df.merge(mission_agg, on="Launch Id", how="left")
# Join vehicle configuration and family metadata using the rocket/config name.
base_modeling_df = base_modeling_df.merge(
    config_features[
        [
            "config_name",
            "config_status",
            "config_price_musd_text_parsed",
            "config_liftoff_thrust",
            "config_payload_leo",
            "config_payload_gto",
            "config_stages",
            "config_strap_ons",
            "config_rocket_height",
            "config_fairing_diameter",
            "config_fairing_height",
            "rocket_family",
            "family_success_rate_pct",
        ]
    ],
    left_on="Rocket Name",
    right_on="config_name",
    how="left",
)
# Join organization metadata using the launch table's organization label.
base_modeling_df = base_modeling_df.merge(
    company_features,
    on="Rocket Organisation",
    how="left",
)

base_modeling_df["Country_Code"] = base_modeling_df["Country_Code"].astype(str).str.upper()
base_modeling_df["location_joined"] = base_modeling_df["Orig_Addr"].notna()
base_modeling_df["mission_joined"] = base_modeling_df["mission_rows"].notna()
base_modeling_df["config_joined"] = base_modeling_df["rocket_family"].notna()
base_modeling_df["company_joined"] = base_modeling_df["company_country"].notna()

# Standardize launch-site naming to a stable modeling and NOAA-join grouping.
base_modeling_df["facility_group"] = base_modeling_df.apply(infer_facility_group, axis=1)
# For U.S. launches we keep the raw launch-site coordinate, while for non-U.S. launches we
# use the combined launch-site coordinate from Locations.csv.
base_modeling_df["facility_lat"] = np.where(
    base_modeling_df["Country_Code"].eq("US"),
    base_modeling_df["Lat"],
    base_modeling_df["Comb Launch Site Lat"],
)
base_modeling_df["facility_lon"] = np.where(
    base_modeling_df["Country_Code"].eq("US"),
    base_modeling_df["Lon"],
    base_modeling_df["Comb Launch Site Lon"],
)
base_modeling_df["facility_coordinate_source"] = np.where(
    base_modeling_df["Country_Code"].eq("US"),
    "Locations.csv raw Lat/Lon",
    "Locations.csv Comb Launch Site Lat/Lon",
)

# Locations.csv places Pacific Spaceport Complex Alaska inland; this override keeps the
# facility coordinate aligned with the Kodiak-area launch site used in the NOAA workflow.
psca_mask = base_modeling_df["facility_group"].eq("Pacific Spaceport Complex Alaska")
base_modeling_df.loc[psca_mask, "facility_lat"] = 57.4350
base_modeling_df.loc[psca_mask, "facility_lon"] = -152.3390
base_modeling_df.loc[psca_mask, "facility_coordinate_source"] = (
    "manual override: Kodiak-area PSCA coordinate; Locations.csv appears inland"
)

# Basic outcome helpers that are useful in almost every later modeling notebook.
base_modeling_df["is_success"] = base_modeling_df["Launch Status"].eq("Success").astype(int)
base_modeling_df["is_failure_or_partial"] = base_modeling_df["Launch Status"].ne("Success").astype(int)
base_modeling_df["is_suborbital"] = base_modeling_df["Launch Suborbital"].astype(str).str.lower().eq("suborbital").astype(int)

# Sort by event time so any later rolling or prelaunch feature engineering starts from a
# stable chronological order.
base_modeling_df = base_modeling_df.sort_values(["launch_time_utc", "Launch Id"]).reset_index(drop=True)

## Join Diagnostics

This checkpoint answers a simple but important question: how complete is the cleaned launch
table after all of the non-weather joins?

If one of the source joins is much weaker than expected, it is better to catch it here
before weather is added and the debugging surface gets larger.

In [27]:
prep_diagnostics = pd.DataFrame(
    [
        {"metric": "total launches", "value": len(base_modeling_df)},
        {"metric": "missing launch timestamps", "value": int(base_modeling_df["launch_time_utc"].isna().sum())},
        {"metric": "unique raw locations", "value": int(base_modeling_df["Location"].nunique())},
        {"metric": "unique facility groups", "value": int(base_modeling_df["facility_group"].nunique())},
        {"metric": "location joins available", "value": int(base_modeling_df["location_joined"].sum())},
        {"metric": "mission joins available", "value": int(base_modeling_df["mission_joined"].sum())},
        {"metric": "config joins available", "value": int(base_modeling_df["config_joined"].sum())},
        {"metric": "company joins available", "value": int(base_modeling_df["company_joined"].sum())},
        {"metric": "launches with facility coordinates", "value": int(base_modeling_df[['facility_lat','facility_lon']].notna().all(axis=1).sum())},
    ]
)

prep_diagnostics

,metric,value
0,total launches,6168
1,missing launch timestamps,0
2,unique raw locations,137
3,unique facility groups,40
4,location joins available,6168
5,mission joins available,5557
6,config joins available,6168
7,company joins available,6168
8,launches with facility coordinates,6168


## Column Selection For Feature Engineering

The full joined table contains intermediate columns that are useful for data prep but not
ideal as the starting point for modeling work. This section selects the cleaned,
launch-level columns we want to carry forward into the feature-engineering stage and saves
that base table to disk.

In [28]:
# Keep the base table explicit so later notebooks start from a stable, intentional schema
# rather than inheriting every intermediate join column by accident.
model_base_columns = [
    "Launch Id",
    "launch_time_utc",
    "launch_date",
    "launch_year",
    "launch_month",
    "launch_month_name",
    "launch_quarter",
    "launch_dayofweek",
    "launch_hour_utc",
    "launch_decade",
    "Launch Status",
    "is_success",
    "is_failure_or_partial",
    "Launch Suborbital",
    "is_suborbital",
    "Rocket Name",
    "Rocket Organisation",
    "company_country",
    "company_ownership",
    "Rocket Price",
    "Rocket Price CPI Adjusted",
    "Rocket Payload to LEO",
    "USD/kg to LEO",
    "USD/kg to LEO CPI Adjusted",
    "Location",
    "Country",
    "Country_Code",
    "Operator",
    "Launch Site",
    "Comb Launch Site",
    "facility_group",
    "Lat",
    "Lon",
    "Launch Site Lat",
    "Launch Site Lon",
    "Comb Launch Site Lat",
    "Comb Launch Site Lon",
    "facility_lat",
    "facility_lon",
    "facility_coordinate_source",
    "payload_count",
    "mission_mass",
    "mission_rows",
    "config_status",
    "config_price_musd_text_parsed",
    "config_liftoff_thrust",
    "config_payload_leo",
    "config_payload_gto",
    "config_stages",
    "config_strap_ons",
    "config_rocket_height",
    "config_fairing_diameter",
    "config_fairing_height",
    "rocket_family",
    "family_success_rate_pct",
    "location_joined",
    "mission_joined",
    "config_joined",
    "company_joined",
]

model_base_columns = [c for c in model_base_columns if c in base_modeling_df.columns]
model_base_df = base_modeling_df[model_base_columns].copy()

# Persist the launch-only modeling base table because it is a useful checkpoint even before
# weather is joined in.
model_base_df.to_csv(OUTPUT_DIR / "launch_modeling_base_table.csv", index=False)

pd.DataFrame(
    {
        "output_file": [str(OUTPUT_DIR / "launch_modeling_base_table.csv")],
        "rows": [len(model_base_df)],
        "columns": [model_base_df.shape[1]],
    }
)

,output_file,rows,columns
0,data_final\derived\launch_modeling_base_table.csv,6168,59


## Available NOAA Weather Files

This section defines which downloaded NOAA/LCD files are currently available and which
facility each file belongs to.

It also defines the fixed local-standard-time offsets used for the weather join. The NOAA
LCD files store local station timestamps rather than UTC launch timestamps, so we convert
launches from UTC into each facility's local standard time before matching.

The local file `lcdv2_DOCUMENTATION.pdf` is the reference for the LCDv2 structure. A few
details from that documentation matter directly for this notebook:

- `STATION` is the 11-character GHCN station identifier
- station-year CSV files are stored in metric/SI units
- `DATE` values are local standard time, and daylight saving time is not used
- bulk LCD station files can contain hourly, daily, monthly, and other observation rows

Those conventions are why the next step deliberately filters toward the strongest hourly
observation stream before doing the launch-weather join.

In [29]:
# Map each downloaded NOAA file to the facility_group used in the cleaned launch table.
FILE_TO_FACILITY = {
    "Alcantara_LC.csv": "Alcântara LC",
    "Baikonur_Cosmodrome.csv": "Baikonur Cosmodrome",
    "Base_Aerea_de_Gando.csv": "Base Aerea de Gando",
    "cape_canaveral_sfs.csv": "Cape Canaveral Space Force Station",
    "china_lake.csv": "China Lake",
    "edwards_afb.csv": "Edwards Air Force Base",
    "Guiana_SC.csv": "Guiana SC",
    "Imam_Khomeini_Spaceport.csv": "Imam Khomeini Spaceport",
    "Jiuquan_Satellite_LC.csv": "Jiuquan Satellite LC",
    "Kapustin_Yar.csv": "Kapustin Yar",
    "kennedy_sc.csv": "Kennedy Space Center",
    "Kiritimati_LA.csv": "Kiritimati LA",
    "Mahia_Peninsula.csv": "Māhia Peninsula",
    "Mojave_Air_and__Space_Port.csv": "Mojave Air and Space Port",
    "Naro_Space_Center.csv": "Naro Space Center",
    "pacific_missile_range_facility.csv": "Pacific Missile Range Facility",
    "Pacific_Spaceport_Complex_Alaska.csv": "Pacific Spaceport Complex Alaska",
    "Palmachim_Airbase.csv": "Palmachim Airbase",
    "Plesetsk_Cosmodrome.csv": "Plesetsk Cosmodrome",
    "RAAF_Woomera_RC.csv": "RAAF Woomera RC",
    "Ronald_Reagan_BMDTS.csv": "Ronald Reagan BMDTS",
    "Satish_Dhawan_SC.csv": "Satish Dhawan SC",
    "Shahrud_MTS.csv": "Shahrud MTS",
    "Sohae_SLS.csv": "Sohae SLS",
    "Svobodny_Cosmodrome.csv": "Svobodny Cosmodrome",
    "Taiyuan_Satellite_LC.csv": "Taiyuan Satellite LC",
    "Tanegashima_SC.csv": "Tanegashima SC",
    "Tonghae_SLG.csv": "Tonghae SLG",
    "Uchinoura_SC.csv": "Uchinoura SC",
    "vandenberg_sfb.csv": "Vandenberg Space Force Base",
    "Vostochny_Cosmodrome.csv": "Vostochny Cosmodrome",
    "wallops_flight_facility.csv": "Wallops Flight Facility",
    "Wenchang_Satellite_LC.csv": "Wenchang Satellite LC",
    "Xichang_Satellite_LC.csv": "Xichang Satellite LC",
    "Yasny_Cosmodrome.csv": "Yasny Cosmodrome",
}

# Launch times are stored in UTC, but NOAA LCD timestamps are local station times. These
# offsets let us convert launches into the same local-time frame before the nearest-time
# join.
FACILITY_UTC_OFFSET_HOURS = {
    "Alcântara LC": -3.0,
    "Baikonur Cosmodrome": 5.0,
    "Base Aerea de Gando": 0.0,
    "Cape Canaveral Space Force Station": -5.0,
    "China Lake": -8.0,
    "Edwards Air Force Base": -8.0,
    "Guiana SC": -3.0,
    "Imam Khomeini Spaceport": 3.5,
    "Jiuquan Satellite LC": 8.0,
    "Kapustin Yar": 4.0,
    "Kennedy Space Center": -5.0,
    "Kiritimati LA": 14.0,
    "Māhia Peninsula": 12.0,
    "Mojave Air and Space Port": -8.0,
    "Naro Space Center": 9.0,
    "Pacific Missile Range Facility": -10.0,
    "Pacific Spaceport Complex Alaska": -9.0,
    "Palmachim Airbase": 2.0,
    "Plesetsk Cosmodrome": 3.0,
    "RAAF Woomera RC": 9.5,
    "Ronald Reagan BMDTS": 12.0,
    "Satish Dhawan SC": 5.5,
    "Shahrud MTS": 3.5,
    "Sohae SLS": 9.0,
    "Svobodny Cosmodrome": 9.0,
    "Taiyuan Satellite LC": 8.0,
    "Tanegashima SC": 9.0,
    "Tonghae SLG": 9.0,
    "Uchinoura SC": 9.0,
    "Vandenberg Space Force Base": -8.0,
    "Vostochny Cosmodrome": 9.0,
    "Wallops Flight Facility": -5.0,
    "Wenchang Satellite LC": 8.0,
    "Xichang Satellite LC": 8.0,
    "Yasny Cosmodrome": 5.0,
}

# The weather join keeps a focused subset of hourly numeric and text fields. This is a base
# weather table for later feature engineering, not the final engineered weather feature set.
WEATHER_NUMERIC_COLUMNS = [
    "HourlyAltimeterSetting",
    "HourlyDryBulbTemperature",
    "HourlyDewPointTemperature",
    "HourlyRelativeHumidity",
    "HourlyPrecipitation",
    "HourlyVisibility",
    "HourlyStationPressure",
    "HourlySeaLevelPressure",
    "HourlyWindSpeed",
    "HourlyWindDirection",
    "HourlyWindGustSpeed",
    "HourlyWetBulbTemperature",
]

WEATHER_TEXT_COLUMNS = [
    "HourlyPresentWeatherType",
    "HourlySkyConditions",
]

SHORT_DURATION_PRECIP_COLUMNS = [
    "HourlyPrecipitation",
    "HourlyPrecipitation01Hour",
    "HourlyPrecipitation03Hour",
    "HourlyPrecipitation06Hour",
]

# Scan the NOAA_data folder and keep only files we know how to map to a facility.
available_noaa_files = pd.DataFrame(
    [
        {
            "file_name": path.name,
            "facility_group": FILE_TO_FACILITY.get(path.name, path.stem),
            "utc_offset_hours": FACILITY_UTC_OFFSET_HOURS.get(FILE_TO_FACILITY.get(path.name, path.stem)),
            "file_path": str(path),
        }
        for path in sorted((DATA_DIR / "NOAA_data").glob("*.csv"))
        if path.name in FILE_TO_FACILITY
    ]
).sort_values("facility_group").reset_index(drop=True)

available_noaa_files

,file_name,facility_group,utc_offset_hours,file_path
0,Alcantara_LC.csv,Alcântara LC,-3.0,data_final\NOAA_data\Alcantara_LC.csv
1,Baikonur_Cosmodrome.csv,Baikonur Cosmodrome,5.0,data_final\NOAA_data\Baikonur_Cosmodrome.csv
2,Base_Aerea_de_Gando.csv,Base Aerea de Gando,0.0,data_final\NOAA_data\Base_Aerea_de_Gando.csv
3,cape_canaveral_sfs.csv,Cape Canaveral Space Force Station,-5.0,data_final\NOAA_data\cape_canaveral_sfs.csv
4,china_lake.csv,China Lake,-8.0,data_final\NOAA_data\china_lake.csv
5,edwards_afb.csv,Edwards Air Force Base,-8.0,data_final\NOAA_data\edwards_afb.csv
6,Guiana_SC.csv,Guiana SC,-3.0,data_final\NOAA_data\Guiana_SC.csv
7,Imam_Khomeini_Spaceport.csv,Imam Khomeini Spaceport,3.5,data_final\NOAA_data\Imam_Khomeini_Spaceport.csv
8,Jiuquan_Satellite_LC.csv,Jiuquan Satellite LC,8.0,data_final\NOAA_data\Jiuquan_Satellite_LC.csv
9,Kapustin_Yar.csv,Kapustin Yar,4.0,data_final\NOAA_data\Kapustin_Yar.csv


## Join Available NOAA Weather To The Launch Base Table

This is the core weather-integration step.

What the code is doing:

1. load a downloaded NOAA/LCD file for one facility
2. clean numeric weather fields that arrive as text-with-units
3. keep the strongest hourly observation at each timestamp
4. convert launch UTC timestamps into facility local standard time
5. join each launch to the nearest hourly weather row within 2 hours

Why it is doing that:

- NOAA LCD files are not guaranteed to be a clean one-row-per-hour table
- the LCD documentation states that station-year timestamps are in local standard time, so
  direct UTC-to-local matching would fail without a timezone conversion
- a nearest-time join with tolerance is a practical way to attach weather while still
  preserving diagnostics about match quality

In [30]:
def clean_lcd_numeric(series: pd.Series) -> pd.Series:
    # LCD numeric fields are often text fields with units or extra symbols, so we normalize
    # them before modeling.
    return pd.to_numeric(
        series.astype(str).str.extract(r"([-+]?[0-9]*\.?[0-9]+)")[0],
        errors="coerce",
    )


def load_best_hourly_weather(path: Path) -> pd.DataFrame:
    # Read the raw LCD file, keep the weather columns we care about, and remove duplicate
    # column names that sometimes appear when similar fields are present in the source file.
    weather_raw = pd.read_csv(path, low_memory=False)
    keep_cols = ["STATION", "DATE", "REPORT_TYPE"] + [
        c
        for c in WEATHER_NUMERIC_COLUMNS + WEATHER_TEXT_COLUMNS + SHORT_DURATION_PRECIP_COLUMNS
        if c in weather_raw.columns
    ]
    keep_cols = list(dict.fromkeys(keep_cols))
    weather = weather_raw[keep_cols].copy()
    weather = weather.loc[:, ~weather.columns.duplicated()].copy()

    for col in [c for c in WEATHER_NUMERIC_COLUMNS + SHORT_DURATION_PRECIP_COLUMNS if c in weather.columns]:
        weather[col] = clean_lcd_numeric(weather[col])

    # NOAA timestamps are already in local station time in these downloaded LCD files.
    weather["weather_obs_time_lstd"] = pd.to_datetime(weather["DATE"], errors="coerce")
    weather = weather.dropna(subset=["weather_obs_time_lstd"]).copy()
    weather["weather_obs_time_lstd"] = weather["weather_obs_time_lstd"].astype("datetime64[ns]")

    numeric_cols = [c for c in WEATHER_NUMERIC_COLUMNS if c in weather.columns]
    text_cols = [c for c in WEATHER_TEXT_COLUMNS if c in weather.columns]

    # Keep the "best" hourly row at each timestamp by preferring the row with the most
    # populated hourly weather fields.
    weather["hourly_nonnulls"] = weather[numeric_cols].notna().sum(axis=1)
    for col in text_cols:
        weather["hourly_nonnulls"] += weather[col].notna().astype(int)

    weather = (
        weather.loc[weather["hourly_nonnulls"] > 0]
        .sort_values(["weather_obs_time_lstd", "hourly_nonnulls"], ascending=[True, False])
        .drop_duplicates(subset=["weather_obs_time_lstd"], keep="first")
        .sort_values("weather_obs_time_lstd")
        .reset_index(drop=True)
    )

    # Create a few immediately useful parsed weather flags directly in the base table.
    weather["present_weather_rain_flag"] = (
        weather.get("HourlyPresentWeatherType", pd.Series("", index=weather.index))
        .fillna("")
        .astype(str)
        .str.contains(r"RA|DZ|SH", regex=True)
    )
    weather["present_weather_fog_flag"] = (
        weather.get("HourlyPresentWeatherType", pd.Series("", index=weather.index))
        .fillna("")
        .astype(str)
        .str.contains(r"FG|BR|HZ", regex=True)
    )
    weather["present_weather_thunder_flag"] = (
        weather.get("HourlyPresentWeatherType", pd.Series("", index=weather.index))
        .fillna("")
        .astype(str)
        .str.contains(r"TS", regex=True)
    )
    weather["cloud_cover_broken_or_overcast_flag"] = (
        weather.get("HourlySkyConditions", pd.Series("", index=weather.index))
        .fillna("")
        .astype(str)
        .str.contains(r"BKN|OVC", regex=True)
    )

    available_short_duration_cols = [c for c in SHORT_DURATION_PRECIP_COLUMNS if c in weather.columns]
    if available_short_duration_cols:
        weather["short_duration_precip_max"] = weather[available_short_duration_cols].max(axis=1, skipna=True)
    else:
        weather["short_duration_precip_max"] = np.nan

    return weather


weather_merge_frames = []
weather_join_coverage_rows = []

# Facilities without a downloaded NOAA file still stay in the final base table; they simply
# carry missing weather fields and a False weather_matched flag.
unmatched_launches = model_base_df.loc[
    ~model_base_df["facility_group"].isin(available_noaa_files["facility_group"])
].copy()
unmatched_launches["launch_time_lstd"] = pd.NaT
unmatched_launches["weather_obs_time_lstd"] = pd.NaT
unmatched_launches["weather_matched"] = False
unmatched_launches["weather_time_diff_minutes"] = np.nan
unmatched_launches["weather_file_name"] = pd.NA
unmatched_launches["weather_station_id"] = pd.NA
weather_merge_frames.append(unmatched_launches)

for _, file_row in available_noaa_files.iterrows():
    facility = file_row["facility_group"]
    utc_offset_hours = file_row["utc_offset_hours"]
    facility_launches = model_base_df.loc[model_base_df["facility_group"] == facility].copy()

    if facility_launches.empty:
        continue

    # Convert launch UTC timestamps into the local standard time frame used by the NOAA
    # station file so merge_asof compares timestamps in the same clock system.
    facility_launches["launch_time_lstd"] = (
        facility_launches["launch_time_utc"] + pd.to_timedelta(utc_offset_hours, unit="h")
    ).dt.tz_localize(None)
    facility_launches["launch_time_lstd"] = facility_launches["launch_time_lstd"].astype("datetime64[ns]")
    facility_launches = facility_launches.sort_values("launch_time_lstd")

    weather = load_best_hourly_weather(Path(file_row["file_path"]))
    weather = weather.rename(columns={"STATION": "weather_station_id"})

    # Match each launch to the nearest weather observation within 2 hours.
    merged = pd.merge_asof(
        facility_launches,
        weather,
        left_on="launch_time_lstd",
        right_on="weather_obs_time_lstd",
        direction="nearest",
        tolerance=pd.Timedelta("2h"),
    )

    merged["weather_matched"] = merged["weather_obs_time_lstd"].notna()
    merged["weather_time_diff_minutes"] = (
        (merged["launch_time_lstd"] - merged["weather_obs_time_lstd"]).dt.total_seconds().abs() / 60
    )
    merged["weather_file_name"] = file_row["file_name"]

    # Store facility-level diagnostics so we can evaluate weather-join quality separately
    # from later modeling work.
    weather_join_coverage_rows.append(
        {
            "facility_group": facility,
            "launches": len(facility_launches),
            "matched_launches": int(merged["weather_matched"].sum()),
            "match_rate": float(merged["weather_matched"].mean()),
            "median_abs_diff_minutes": merged.loc[merged["weather_matched"], "weather_time_diff_minutes"].median(),
            "weather_file_name": file_row["file_name"],
            "weather_start_lstd": weather["weather_obs_time_lstd"].min() if len(weather) else pd.NaT,
            "weather_end_lstd": weather["weather_obs_time_lstd"].max() if len(weather) else pd.NaT,
            "weather_rows_after_cleaning": len(weather),
        }
    )

    weather_merge_frames.append(merged)

# Recombine all facilities, both matched and unmatched, into one modeling-ready table.
model_base_weather_df = pd.concat(weather_merge_frames, ignore_index=True)
model_base_weather_df = model_base_weather_df.sort_values(["launch_time_utc", "Launch Id"]).reset_index(drop=True)

weather_join_coverage = pd.DataFrame(weather_join_coverage_rows).sort_values(
    ["matched_launches", "match_rate"], ascending=[False, False]
).reset_index(drop=True)

weather_join_coverage

,facility_group,launches,matched_launches,match_rate,median_abs_diff_minutes,weather_file_name,weather_start_lstd,weather_end_lstd,weather_rows_after_cleaning
0,Plesetsk Cosmodrome,1648,1091,0.662015,45.0,Plesetsk_Cosmodrome.csv,1966-01-01 00:00:00,1988-10-08 15:00:00,57732
1,Vandenberg Space Force Base,711,672,0.945148,13.0,vandenberg_sfb.csv,1959-01-01 00:00:00,2021-12-31 23:58:00,624228
2,Cape Canaveral Space Force Station,810,509,0.628395,14.0,cape_canaveral_sfs.csv,1957-12-01 00:00:00,2021-12-31 23:56:00,459992
3,Guiana SC,311,304,0.977492,11.0,Guiana_SC.csv,1972-12-31 23:00:00,2021-12-31 23:30:00,483251
4,Kennedy Space Center,192,173,0.901042,0.0,kennedy_sc.csv,1978-03-16 19:00:00,2021-12-31 23:56:00,1978955
5,Xichang Satellite LC,167,167,1.000000,53.0,Xichang_Satellite_LC.csv,1984-01-01 02:00:00,2021-12-31 23:00:00,110869
6,Baikonur Cosmodrome,1525,145,0.095082,50.0,Baikonur_Cosmodrome.csv,1966-01-01 02:00:00,1991-12-21 02:00:00,13176
7,Tanegashima SC,85,67,0.788235,21.0,Tanegashima_SC.csv,1975-01-01 08:00:00,2021-12-31 18:00:00,201486
8,Kapustin Yar,97,54,0.556701,58.0,Kapustin_Yar.csv,1961-01-01 03:00:00,1974-03-10 00:00:00,23264
9,Wallops Flight Facility,49,35,0.714286,13.0,wallops_flight_facility.csv,1966-10-01 07:00:00,2021-12-31 23:54:00,397796


## Save The Launch Base Table With Available NOAA Weather

This checkpoint writes out:

- the launch-level base table with whatever NOAA weather is currently available
- a facility-level coverage table showing how well the weather join worked

Saving both is useful because future modeling notebooks usually need the row-level table,
while troubleshooting and data-quality review usually start from the facility-level summary.

In [31]:
# Persist both the row-level modeling table and the facility-level weather join diagnostics.
model_base_weather_df.to_csv(
    OUTPUT_DIR / "launch_modeling_base_with_weather.csv",
    index=False,
)
weather_join_coverage.to_csv(
    OUTPUT_DIR / "launch_modeling_weather_join_coverage.csv",
    index=False,
)

pd.DataFrame(
    {
        "output_file": [
            str(OUTPUT_DIR / "launch_modeling_base_with_weather.csv"),
            str(OUTPUT_DIR / "launch_modeling_weather_join_coverage.csv"),
        ],
        "rows": [
            len(model_base_weather_df),
            len(weather_join_coverage),
        ],
    }
)

,output_file,rows
0,data_final\derived\launch_modeling_base_with_weather.csv,6168
1,data_final\derived\launch_modeling_weather_join_coverage.csv,35


## Modeling Base Dataframe With Available NOAA Weather

This is the cleaned launch-level dataframe with all currently available NOAA weather joined in.
It is ready for the next step of feature engineering.

The final table is intentionally broad:

- launch-side fields remain available for engineering maturity, cadence, and reliability
  features
- raw joined weather columns remain available for engineering site-relative and rolling
  weather features
- join-quality fields such as `weather_matched` and `weather_time_diff_minutes` remain
  available so later modeling work can decide how strict to be about weather quality

In [32]:
model_base_weather_df

,Launch Id,launch_time_utc,launch_date,launch_year,launch_month,launch_month_name,launch_quarter,launch_dayofweek,launch_hour_utc,launch_decade,Launch Status,is_success,is_failure_or_partial,Launch Suborbital,is_suborbital,Rocket Name,Rocket Organisation,company_country,company_ownership,Rocket Price,Rocket Price CPI Adjusted,Rocket Payload to LEO,USD/kg to LEO,USD/kg to LEO CPI Adjusted,Location,Country,Country_Code,Operator,Launch Site,Comb Launch Site,facility_group,Lat,Lon,Launch Site Lat,Launch Site Lon,Comb Launch Site Lat,Comb Launch Site Lon,facility_lat,facility_lon,facility_coordinate_source,payload_count,mission_mass,mission_rows,config_status,config_price_musd_text_parsed,config_liftoff_thrust,config_payload_leo,config_payload_gto,config_stages,config_strap_ons,config_rocket_height,config_fairing_diameter,config_fairing_height,rocket_family,family_success_rate_pct,location_joined,mission_joined,config_joined,company_joined,launch_time_lstd,weather_obs_time_lstd,weather_matched,weather_time_diff_minutes,weather_file_name,weather_station_id,DATE,REPORT_TYPE,HourlyDryBulbTemperature,HourlyDewPointTemperature,HourlySeaLevelPressure,HourlyWindSpeed,HourlyWindDirection,hourly_nonnulls,present_weather_rain_flag,present_weather_fog_flag,present_weather_thunder_flag,cloud_cover_broken_or_overcast_flag,short_duration_precip_max,HourlyStationPressure,HourlyRelativeHumidity,HourlyAltimeterSetting,HourlyPrecipitation,HourlyVisibility,HourlyWindGustSpeed,HourlyWetBulbTemperature,HourlyPresentWeatherType,HourlySkyConditions
0,590,1957-10-04 19:28:00+00:00,1957-10-04,1957,10,October,4,4,19,1950,Success,1,0,Orbital,0,Sputnik 8K71PS,RVSN USSR,Russia,State,NaN,NaN,510.0,NaN,NaN,"Site 1/5, Baikonur Cosmodrome, Kazakhstan",Kazakhstan,KZ,Russian Aerospace Forces,Baikonur Cosmodrome,Baikonur Cosmodrome,Baikonur Cosmodrome,45.964585,63.305243,45.964585,63.305243,45.964585,63.305243,45.964585,63.305243,Locations.csv Comb Launch Site Lat/Lon,1.0,84.0,1.0,Retired,NaN,4.0,510.0,0.0,2.0,4.0,30.20,2.95,4.00,Sputnik 8K71PS,100.0,True,True,True,True,1957-10-05 00:28:00,NaT,False,NaN,Baikonur_Cosmodrome.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,593,1957-11-03 02:30:00+00:00,1957-11-03,1957,11,November,4,6,2,1950,Success,1,0,Orbital,0,Sputnik 8K71PS,RVSN USSR,Russia,State,NaN,NaN,510.0,NaN,NaN,"Site 1/5, Baikonur Cosmodrome, Kazakhstan",Kazakhstan,KZ,Russian Aerospace Forces,Baikonur Cosmodrome,Baikonur Cosmodrome,Baikonur Cosmodrome,45.964585,63.305243,45.964585,63.305243,45.964585,63.305243,45.964585,63.305243,Locations.csv Comb Launch Site Lat/Lon,1.0,508.0,1.0,Retired,NaN,4.0,510.0,0.0,2.0,4.0,30.20,2.95,4.00,Sputnik 8K71PS,100.0,True,True,True,True,1957-11-03 07:30:00,NaT,False,NaN,Baikonur_Cosmodrome.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,845,1957-12-06 16:44:00+00:00,1957-12-06,1957,12,December,4,4,16,1950,Failure,0,1,Orbital,0,Vanguard,US Navy,USA,State,NaN,NaN,10.0,NaN,NaN,"LC-18A, Cape Canaveral SFS, Florida, USA",United States,US,United States Space Force,Cape Canaveral SFS,Cape Canaveral SFS/Kennedy SC,Cape Canaveral Space Force Station,28.448986,-80.561827,28.489690,-80.563003,28.501081,-80.567792,28.448986,-80.561827,Locations.csv raw Lat/Lon,1.0,2.0,1.0,Retired,NaN,135.0,10.0,NaN,3.0,0.0,23.00,NaN,NaN,Vanguard,22.7,True,True,True,True,1957-12-06 11:44:00,1957-12-06 12:00:00,True,16.0,cape_canaveral_sfs.csv,USW00012868,1957-12-06T12:00:00,SAO,70.0,53.0,30.37,16.0,110.0,11.0,False,False,False,False,NaN,30.35,55.0,NaN,NaN,9.94,NaN,60.0,||00,SCT:03 44
3,844,1958-02-01 03:47:00+00:00,1958-02-01,1958,2,February,1,5,3,1950,Success,1,0,Orbital,0,Juno I,ABMA,USA,State,NaN,NaN,11.0,NaN,NaN,"LC-26A, Cape Canaveral SFS, Florida, USA",United States,US,United States Space Force,Cape Canaveral SFS,Cape Canaveral SFS/Kennedy SC,Cape Canaveral Space Force Station,28.439421,-80.573307,28.489690,-80.563003,28.501081,-80.567792,28.439421,-8

## Assumptions and Caveats

This notebook is intended to produce a practical launch-plus-weather base table, not a
final audited scientific data product. A few assumptions and limitations should stay
visible before feature engineering and modeling begin.

### Launch-side assumptions

- The join from `Launches.csv` to `Locations.csv` assumes the raw `Location` string is the
  correct key for site metadata.
- The join from `Launches.csv` to `Configs.csv` assumes `Rocket Name` matches `Config`
  closely enough to use as a direct key.
- The facility grouping logic is intentionally opinionated for major U.S. sites so that the
  launch table lines up with the NOAA workflow. If the project later changes its site
  definitions, this grouping may need to be revised.
- `Pacific Spaceport Complex Alaska` uses a manual coordinate override because the location
  coordinates in `Locations.csv` appear inland and are not suitable for station-distance
  work.

### NOAA / LCD assumptions

- The NOAA join only uses files currently available in `data_final/NOAA_data`. If a file is
  missing, the launch remains in the base table but weather fields will be null.
- LCD timestamps are treated as local standard time, consistent with the LCDv2
  documentation. That means each facility requires a fixed UTC offset before time matching.
- The fixed UTC offsets in this notebook are a pragmatic approximation. They are good
  enough for the current join workflow, but they are still a simplification of real-world
  timezone history.
- The downloaded NOAA files can contain hourly, daily, monthly, and other rows together, so
  the join logic keeps the strongest hourly row at each timestamp rather than trusting the
  raw file structure directly.
- A launch is counted as weather-matched only if a weather observation is found within
  2 hours. That threshold is a project choice, not a NOAA requirement.

### Modeling caveats

- Not every launch has weather data, and not every facility has comparable weather coverage.
  Any future model that uses weather should treat `weather_matched`,
  `weather_time_diff_minutes`, and missingness patterns as meaningful signals.
- The weather fields in the base table are still relatively raw. They are suitable for
  feature engineering, but not necessarily for immediate direct modeling without additional
  quality review.
- Join quality differs a lot by facility. Downstream modeling work should check whether a
  site-specific or coverage-aware modeling strategy is more appropriate than treating all
  weather-matched launches as equally reliable.